# Immortalite — Self-Play Training (Colab)

Trains the lightweight self-play chess engine on a free GPU. Checkpoints are written to Google Drive so the run survives disconnects.

**Steps:** set runtime to GPU (Runtime -> Change runtime type -> GPU), then run the cells top to bottom.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite'):
    !git clone https://github.com/carlo-wong/immortalite.git
%cd /content/immortalite
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy tqdm

In [ ]:
# 3. Mount Google Drive for crash-proof checkpoints.
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/immortalite_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 4. Confirm GPU is available.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 5. Train with the GPU preset. Resumes automatically from the Drive checkpoint.
# Keep a snapshot every 10 iterations in addition to latest.pt.
# Enable periodic strength gates (current net vs prior snapshot).
import os
resume = os.path.join(CKPT_DIR, 'latest.pt')
resume_arg = f'--resume {resume}' if os.path.exists(resume) else ''
!python -m engine.train --iterations 50 --device cuda --gpu --save-every 10 --gate-every 10 --gate-games 20 --checkpoint-dir "$CKPT_DIR" $resume_arg

In [ ]:
# 6. Plot training progress (run AFTER at least one training iteration finishes).
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 5 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            fig, ax = plt.subplots(2, 3, figsize=(16, 8))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if 'winrate_vs_prev' not in df.columns:
                ax[5].text(0.5, 0.5, 'winrate_vs_prev missing', ha='center', va='center')
                ax[5].set_title('Strength gate (vs prev net)')
                ax[5].set_xlabel('iteration')
            else:
                gate_df = df.dropna(subset=['winrate_vs_prev'])
                if gate_df.empty:
                    ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                    ax[5].set_title('Strength gate (vs prev net)')
                    ax[5].set_xlabel('iteration')
                else:
                    gate_df.plot(x='iter', y='winrate_vs_prev', marker='o', ax=ax[5], title='Strength gate (vs prev net)')
                    ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)

            plt.tight_layout()
            plt.show()

## After training
Download `latest.pt` from your Drive folder and run the analysis server locally:

```bash
set IMMORTALITE_CHECKPOINT=path\to\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```
Then open http://localhost:8000/app/ .